# MNIST Digit Recognition — CNN Training Notebook

This notebook loads the MNIST handwritten-digit dataset, preprocesses it, trains a CNN, evaluates it on the held-out test set, and saves the model plus plots used by the Streamlit app. Run cells from top to bottom.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)


In [ ]:
# Reproducibility settings
np.random.seed(42)
tf.random.set_seed(42)

# Resolve the project root so files are saved inside the repository.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())


In [ ]:
# Load the MNIST handwritten-digit dataset (60,000 training, 10,000 test images)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape, "Training labels:", y_train.shape)
print("Test images:", x_test.shape, "Test labels:", y_test.shape)


In [ ]:
# Scale grayscale pixel values from [0, 255] to [0, 1] and add the CNN channel dimension.
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Preprocessed training shape:", x_train.shape)
print("Preprocessed test shape:", x_test.shape)


In [ ]:
# View a few training examples
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for index, ax in enumerate(axes):
    ax.imshow(x_train[index].squeeze(), cmap="gray")
    ax.set_title(f"Label: {y_train[index]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Define the CNN architecture. This matches the included digit_cnn.keras model.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
# Train for six epochs to match the six points shown in the included training-history plot.
history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=6,
    batch_size=128,
    verbose=1,
)


In [ ]:
# Evaluate using the held-out MNIST test set
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")

probabilities = model.predict(x_test, batch_size=256, verbose=0)
y_pred = np.argmax(probabilities, axis=1)
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# Save the trained model for app.py
model_path = MODEL_DIR / "digit_cnn.keras"
model.save(model_path)
print(f"Model saved to: {model_path}")


In [ ]:
# Save training and validation accuracy/loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

fig.tight_layout()
history_path = RESULTS_DIR / "training_history.png"
fig.savefig(history_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Training history saved to: {history_path}")


In [ ]:
# Generate and save the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=np.arange(10))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10)).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=True
)
ax.set_title("MNIST Confusion Matrix")
fig.tight_layout()
cm_path = RESULTS_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Confusion matrix saved to: {cm_path}")


In [ ]:
# Inspect and save a few examples the model misclassified
misclassified = np.flatnonzero(y_pred != y_test)
number_to_show = min(15, len(misclassified))

if number_to_show:
    fig, axes = plt.subplots(3, 5, figsize=(10, 6))
    for ax in axes.flat:
        ax.axis("off")
    for ax, index in zip(axes.flat, misclassified[:number_to_show]):
        ax.imshow(x_test[index].squeeze(), cmap="gray")
        ax.set_title(f"True: {y_test[index]} | Pred: {y_pred[index]}")
        ax.axis("off")
    fig.tight_layout()
    mistakes_path = RESULTS_DIR / "misclassified_examples.png"
    fig.savefig(mistakes_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Misclassified examples saved to: {mistakes_path}")
else:
    print("No misclassified examples in this test run.")


## Completion checklist

After all cells finish successfully, the repository should contain:

- `models/digit_cnn.keras`
- `results/training_history.png`
- `results/confusion_matrix.png`
- `results/misclassified_examples.png`

The app can then be started from the repository root with `streamlit run app.py`.
